#### Import module

In [2]:
import pandas as pd
from pyproj import Transformer
import requests
import json
from bs4 import BeautifulSoup
from tqdm import tqdm
import time
import folium

#### 공장 데이터 전처리

In [ ]:
def get_location_from_kakao(address, api_key='b5d48d5493392925bac2082b4545b719'):

    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {api_key}"}
    params = {"query": address}

    response = requests.get(url, headers=headers, params=params)

    data = response.json()

    if data['documents'] == []:
        lat, long = '',''
    else:
        long = data['documents'][0]['x']
        lat = data['documents'][0]['y']
    

    return [lat, long]


def get_df_factory(alpha=1):
    '''
    alpha는 가중치 점수
    '''
    df = pd.read_csv('data/factory/한국산업단지공단_전국등록공장현황_20200229.csv', encoding='cp949')

    # 시도명 필터링
    df_seoul_factory = df[df['시도명'] == '서울특별시']

    # 소기업 필터링
    df_seoul_factory = df_seoul_factory[df_seoul_factory['공장규모'] != '소기업']

    # 업종 필터링
    target_codes = (
        '13',  # 섬유제품 제조업 (원단, 텐트, 위장막 등 원자재 섬유류)
        '14',  # 의복, 의복액세서리 및 모피제품 제조업 (군복, 근무복, 방한복 등 피복류)
        '15',  # 가죽, 가방 및 신발 제조업 (전투화, 배낭, 개인 장구류)
        '19',  # 코크스, 연탄 및 석유정제품 제조업 (석유, 석탄 등 에너지원)
        '20',  # 화학물질 및 화학제품 제조업 (화공품, 화약 및 폭발물)
        '23',  # 비금속 광물제품 제조업 (시멘트, 콘크리트 등 건설/축성 자재)
        '24',  # 1차 금속 제조업 (철강, 제련 등 광물 및 금속 원자재)
        '25',  # 금속가공제품 제조업 (무기, 총포탄 생산 및 각종 금속 부품)
        '26',  # 전자부품, 컴퓨터, 영상 및 통신장비 제조업 (군용 통신 장비, 레이더 등)
        '29',  # 기타 기계 및 장비 제조업 (취사기구, 수리 부속, 일반 기계)
        '31'   # 기타 운송장비 제조업 (선박, 항공기, 장갑차 등 군용 장비 제조 및 수리)
    )

    df_factory_filtered = df_seoul_factory[
        df_seoul_factory['대표업종'].astype(str).str.startswith(target_codes, na=False)
    ]


    df_factory_filtered = df_factory_filtered.drop_duplicates(subset=['공장주소'])
    df_factory_filtered = df_factory_filtered.reset_index()
    df_factory_filtered = df_factory_filtered.drop(columns='index')
    df_factory_filtered

    addresses = df_factory_filtered['공장주소'].values
    lats = []
    longs = []

    for i, addr in enumerate(tqdm(addresses)):
        # API 과부하 및 차단 방지를 위해 아주 짧은 휴식 시간을 준다 (선택 사항)
        time.sleep(0.05) 
        
        # 이전에 만든 API 호출 함수 사용
        loc = get_location_from_kakao(addr)
        lats.append(loc[0])
        longs.append(loc[1])


    df_factory = pd.DataFrame()
    df_factory['name'] = df_factory_filtered['회사명']
    df_factory['latitude'] = lats
    df_factory['longitude'] = longs
    df_factory['tag'] = pd.Series(['산업시설']*len(df_factory))
    df_factory['score'] = pd.Series([alpha]*len(df_factory))

    # type 변경
    df_factory['latitude'] = pd.to_numeric(df_factory['latitude'], errors='coerce')
    df_factory['longitude'] = pd.to_numeric(df_factory['longitude'], errors='coerce')
    
    # 결측치 제거
    df_factory = df_factory.dropna(subset=['latitude', 'longitude'])

    return df_factory

In [88]:
df_factory = get_df_factory(alpha=1)
df_factory.to_csv('data/factory/df_factory.csv', index=False)

C:\Users\user2\AppData\Local\Temp\ipykernel_13452\3854174735.py:25: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/factory/한국산업단지공단_전국등록공장현황_20200229.csv', encoding='cp949')
100%|██████████| 216/216 [00:24<00:00,  8.71it/s]


#### 변전소 데이터 전처리

In [89]:
def get_df_substation(alpha=1):
    df = pd.read_csv('data/substation/변전소 위치(1차).csv',encoding='utf-8')
    df = df.drop(columns='유형')
    df.columns = ['name', 'latitude','longitude']
    df['tag'] = ['변전소'] * len(df)
    df['score'] = [alpha]*len(df)
    df_substation = df
    return df_substation

In [90]:
df_substation = get_df_substation()
df_substation.to_csv('data/substation/df_substation.csv', index=False)

#### 교량 데이터 전처리

In [91]:
def get_df_bridge(alpha=1):

    df = pd.read_csv('data/bridge/교량 위치.csv')

    df = df.drop(columns='Unnamed: 0')
    df = df.drop(columns='소재지지번주소')

    df.columns = ['name','latitude','longitude']
    df['tag'] = ['교량'] * len(df)
    df['score'] = [alpha]*len(df)
    df_bridge = df
    return df_bridge

In [92]:
df_bridge = get_df_bridge()
df_bridge.to_csv('data/bridge/df_bridge.csv', index=False)

#### 병원 데이터 전처리

In [93]:
def get_df_hospital(alpha=1):
    '''
    alpha는 가중치 점수
    '''
    df = pd.read_csv('data/hospital/서울시 병의원 위치 정보.csv', encoding='utf-8')

    # 약국 제거
    condition = (df['병원분류명'] == '기타') & (df['기관명'].str.contains('약국'))
    df = df[~condition]


    # 중요 병원 필터링
    df_filtered = df[df['병원분류명'].isin(['병원','종합병원','보건소','기타'])]

    # 중요 건물 데이터 베이스에 추가
    df_hospital = pd.DataFrame()
    df_hospital['name'] = df_filtered['기관명']
    df_hospital['latitude'] = df_filtered['병원위도']
    df_hospital['longitude'] = df_filtered['병원경도']
    df_hospital = df_hospital.reset_index()
    df_hospital = df_hospital.drop(columns='index')

    df_hospital['tag'] = pd.Series(['병원']*len(df_filtered))
    df_hospital['score'] = pd.Series([alpha]*len(df_filtered))
    return df_hospital

In [94]:
df_hospital = get_df_hospital()
df_hospital.to_csv('data/hospital/df_hospital.csv', index=False)

#### 국가 핵심기반 시설(행안부) 데이터 전처리

In [ ]:
def get_df_core(alpha=1):

    df_core = pd.read_csv('data/core/서울 국가핵심기반 위치.csv')
    df_core = df_core.drop(columns=['Unnamed: 0', '소재지(주사무실)', '관리기관', '지정일자'])
    df_core['tag'] = df_core.iloc[:,0]
    df_core = df_core.drop(columns='분 야 별')
    df_core.columns = ['name', 'latitude', 'longitude', 'tag']
    df_core['score'] = [alpha]*len(df_core)

    # 중복 병원 데이터 제거
    df_core = df_core[df_core['tag'] != '응급의료']
    
    return df_core


In [ ]:
df_core

In [ ]:


def core_sort():
    
    df_core=pd.read_csv('data/core/df_core.csv', index=False)
    tag_list=df_core['tag'].unique()
    tag_list=tag_list.tolist()



    for tag in tag_list:
        df=df_core[df_core['tag']==tag]
        df.to_csv('data/core/df_core_{tag}.csv'.format(tag=tag), index=False)


NameError: name 'df_core' is not defined

#### 기지국 데이터 전처리

In [84]:
df_freq = pd.read_csv('data/frequency/frequency.csv')

In [85]:
df_freq

,Unnamed: 0,허가번호,경도,위도,주파수 대역\n(MHz),구분
0,24,322014110004540,126.981461,127.175308,2600,기지국
1,25,322014110004990,126.986564,127.175308,2600,기지국
2,26,322014110005714,126.982014,127.175308,2600,기지국
3,27,322014110006610,126.976764,127.175308,2600,기지국
4,57,322015110032270,126.983433,127.175308,2600,기지국
...,...,...,...,...,...,...
27158,210995,322022110053001,127.174992,127.175308,2600,기지국
27159,211006,322023110006960,127.180961,127.175308,2600,기지국
27160,211007,322023110006961,127.180622,127.175308,2600,기지국
27161,211008,322023110006962,127.181236,127.175308,2600,기지국


In [81]:
df_freq = df_freq.drop(columns=['Unnamed: 0','주파수 대역\n(MHz)','허가번호'])

In [82]:
df_freq['name'] = [f'기지국{i}' for i in range(len(df_freq))]

In [83]:
df_freq

,경도,위도,구분,name
0,126.981461,127.175308,기지국,기지국0
1,126.986564,127.175308,기지국,기지국1
2,126.982014,127.175308,기지국,기지국2
3,126.976764,127.175308,기지국,기지국3
4,126.983433,127.175308,기지국,기지국4
...,...,...,...,...
27158,127.174992,127.175308,기지국,기지국27158
27159,127.180961,127.175308,기지국,기지국27159
27160,127.180622,127.175308,기지국,기지국27160
27161,127.181236,127.175308,기지국,기지국27161


In [67]:
df_freq.columns = ['longitude', 'latitude', 'tag', 'name']
df_freq = df_freq[['name','latitude','longitude','tag']]

In [ ]:
df_freq['score'] = [1] * len(df_freq)

,name,latitude,longitude,tag
0,기지국0,127.175308,126.981461,기지국
1,기지국1,127.175308,126.986564,기지국
2,기지국2,127.175308,126.982014,기지국
3,기지국3,127.175308,126.976764,기지국
4,기지국4,127.175308,126.983433,기지국
...,...,...,...,...
27158,기지국27158,127.175308,127.174992,기지국
27159,기지국27159,127.175308,127.180961,기지국
27160,기지국27160,127.175308,127.180622,기지국
27161,기지국27161,127.175308,127.181236,기지국


In [73]:
df_freq_test = df_freq.iloc[0:100]

In [ ]:
latsdf_freq_test['latitude'].values


array([127.175308, 127.175308, 127.175308, 127.175308, 127.175308,
       127.175308, 127.175308, 127.175308, 127.175308, 127.175308,
       127.175308, 127.175308, 127.175308, 127.175308, 127.175308,
       127.175308, 127.175308, 127.175308, 127.175308, 127.175308,
       127.175308, 127.175308, 127.175308, 127.175308, 127.175308,
       127.175308, 127.175308, 127.175308, 127.175308, 127.175308,
       127.175308, 127.175308, 127.175308, 127.175308, 127.175308,
       127.175308, 127.175308, 127.175308, 127.175308, 127.175308,
       127.175308, 127.175308, 127.175308, 127.175308, 127.175308,
       127.175308, 127.175308, 127.175308, 127.175308, 127.175308,
       127.175308, 127.175308, 127.175308, 127.175308, 127.175308,
       127.175308, 127.175308, 127.175308, 127.175308, 127.175308,
       127.175308, 127.175308, 127.175308, 127.175308, 127.175308,
       127.175308, 127.175308, 127.175308, 127.175308, 127.175308,
       127.175308, 127.175308, 127.175308, 127.175308, 127.175

In [ ]:
m = folium.Map(location=[37, 127], zoom_start=10)
m.save('data/test_map.html')

d
folium.Marker(location=)